# SentinelAI — Notebook 08: Pipeline Debugging & Lessons Learned

**Purpose:** Document what was built, what broke, why it broke, and how it was fixed.

This notebook is not runnable code — it is a technical diary of the debugging process for the hybrid search pipeline (Notebooks 05-07). Understanding *why* things fail is as important as building them.

---

## Overview: What Was Built

Three Python modules were created for Module 2 (MedDRA Coding Engine):

| File | Purpose |
|------|---------|
| `src/vigilex/coding/hybrid_search.py` | Stage 1: BM25 + Vector search + RRF fusion |
| `src/vigilex/coding/reranker.py` | Stage 2: CrossEncoder reranker (MiniLM-L-6) |
| `src/vigilex/coding/llm_coder.py` | Stage 3: LLM final coding via Ollama |
| `scripts/smoke_test_pipeline.py` | End-to-end smoke test for all three stages |
| `scripts/diagnose_embeddings.py` | Diagnostic script for embedding compatibility |

Plus three companion notebooks (05, 06, 07) with explained walkthroughs.

---
## Part 1: What Each File Does

### `hybrid_search.py`

**Purpose:** Given a free-text MAUDE narrative, retrieve the top-K most relevant MedDRA PTs.

**Two retrieval arms:**

**Arm 1 — BM25 (lexical)** via `pg_trgm`:
- Searches `processed.meddra_terms.pt_name` AND `processed.meddra_llt.llt_name`
- Uses `word_similarity(lower(pt_name), lower(narrative))` — measures how well the short PT name appears as a word-sequence inside the longer narrative
- Why `word_similarity` and not `similarity`? See Bug #3 below.
- Why `lower()`? See Bug #4 below.

**Arm 2 — Vector (semantic)** via `pgvector`:
- Encodes the FIRST SENTENCE of the narrative with PubMedBERT → 768-dim vector
- Queries `pt_embedding` column using cosine distance (`<=>` operator)
- Uses IVFFlat index with `probes=100` for full-scan accuracy
- Why first sentence only? See Bug #5 below.

**Fusion:** Weighted Reciprocal Rank Fusion (RRF)
- `score(d) = 0.4 / (60 + rank_bm25(d)) + 0.6 / (60 + rank_vector(d))`
- Documents not retrieved by an arm get no contribution from that arm

---

### `reranker.py`

**Purpose:** Take the Top-20 RRF results and score each (query, PT) pair with a CrossEncoder.

**Key class:** `CrossEncoderReranker`
- Model: `cross-encoder/ms-marco-MiniLM-L-6-v2` (~90MB)
- Input: list of `SearchResult` objects from `HybridSearcher.search()`
- Output: list of `RerankedResult` objects sorted by `crossencoder_score` descending
- The CrossEncoder sees both query and PT name together → joint attention → better relevance signal

**Key design decision:** The reranker uses the FULL narrative (not just first sentence) because it is a CrossEncoder — it processes both texts jointly and can focus on the relevant parts.

---

### `llm_coder.py`

**Purpose:** Use Ollama (llama3.2) to select the single best PT from the Top-5 reranked candidates.

**Key class:** `LLMCoder`
- Sends a structured prompt to Ollama's `/api/chat` endpoint
- System prompt defines task, rules, and JSON output schema
- Temperature 0.1 for near-deterministic output
- Parses JSON response, validates required fields
- Fallback: if JSON parsing fails → top CrossEncoder candidate with confidence 0.3
- Cases with `confidence < 0.5` are flagged for human review

**Why local Ollama and not OpenAI/Groq?**  
MAUDE data may contain PHI (Protected Health Information). Sending it to external APIs would violate Privacy-by-Design principles. Ollama runs entirely on the Hetzner server — no data leaves the infrastructure.

---

### `smoke_test_pipeline.py`

**Purpose:** Quick automated test that runs all three stages against the live DB and checks results.

**Usage:**
```bash
python scripts/smoke_test_pipeline.py           # full test
python scripts/smoke_test_pipeline.py --skip-llm # skip Ollama stage
python scripts/smoke_test_pipeline.py --verbose  # show detailed results
```

**What it checks:**
- DB connection and row counts (27,361 PT embeddings, 81,719 LLTs)
- Hybrid search returns 10 results with non-zero scores
- Expected PT ("hypoglycaem") appears in top 10
- CrossEncoder returns 5 results sorted by score descending
- LLM returns valid JSON with pt_code, confidence, rationale

**Fails fast:** aborts at first failing check with a hint about the likely cause.

---
## Part 2: Bugs Found and Fixed

### Bug #1 — Port 5432 conflict (local PostgreSQL)

**Symptom:** `FATAL: password authentication failed for user 'vigilex'` even with correct password.

**Root cause:**  
A local Docker container (`ny-taxi-db`, from a previous project) was running PostgreSQL on port 5432. The SSH tunnel also tried to use port 5432. The local container intercepted the connection before it could reach the tunnel.

**How diagnosed:** `netstat -ano | findstr :5432` showed two processes on port 5432 — one was `ssh` (the tunnel), the other was `com.docker.backend` (local container).

**Fix:** `docker stop ny-taxi-db`

**Lesson:** Always check for port conflicts before debugging authentication errors. A `password authentication failed` error does not always mean the password is wrong — it can mean you're connecting to the wrong server.

---

### Bug #2 — `SET ivfflat.probes` via `nextset()` not supported

**Symptom:** `psycopg2.NotSupportedError: not supported by PostgreSQL`

**Root cause:**  
The `SET ivfflat.probes = 10` statement was combined with the SELECT in a single `execute()` call (multi-statement). psycopg2's `nextset()` method (to skip the SET result and fetch the SELECT result) is not supported by the libpq driver.

**Fix:** Split into two separate `cur.execute()` calls:
```python
cur.execute("SET ivfflat.probes = 100")
cur.execute(sql, params)
return cur.fetchall()
```
Both calls run in the same cursor and same transaction, so the SET takes effect for the SELECT.

**Lesson:** psycopg2 does not support multi-statement queries or `nextset()`. Always split statements.

---

### Bug #3 — Wrong similarity function: `similarity()` vs `word_similarity()`

**Symptom:** BM25 arm returned no hypoglycaemia-related results even though "hypoglycaemia" appeared literally in the narrative.

**Root cause:**  
`similarity(a, b)` in pg_trgm computes **Jaccard similarity** between the trigram sets of the two complete strings. For a short PT name (`"Hypoglycaemia"`, ~13 trigrams) compared against a long narrative (`"Patient experienced hypoglycaemia..."`, ~200 trigrams), the Jaccard score is tiny — the shared trigrams are a small fraction of the large narrative trigram set.

**Example:**
```
similarity('Hypoglycaemia', 'Patient experienced hypoglycaemia after...')
→ ~0.05  (too low, filtered out)

word_similarity('Hypoglycaemia', 'Patient experienced hypoglycaemia after...')
→ ~0.85  (correct: measures best local match within the narrative)
```

`word_similarity(a, b)` finds the **best contiguous match** of string `a` within string `b`. This is exactly what we need: does the PT name appear as a word or phrase within the narrative?

**Fix:** Replace `similarity()` with `word_similarity()` in both the WHERE clause and ORDER BY.

**Lesson:** For asymmetric text matching (short term vs. long document), always use `word_similarity()` in pg_trgm, not `similarity()`.

---

### Bug #4 — pg_trgm case-sensitivity

**Symptom:** `word_similarity('Hypoglycaemia', '...hypoglycaemia...')` returned lower scores than expected.

**Root cause:**  
pg_trgm trigrams are **case-sensitive by default**. The trigrams of `"Hypoglycaemia"` start with `" Hy"`, `"Hyp"` while the narrative contains `"hypoglycaemia"` whose trigrams start with `" hy"`, `"hyp"`. These are different trigrams, reducing the similarity score.

**Fix:** Apply `lower()` to both strings before computing similarity:
```sql
word_similarity(lower(pt_name), lower(narrative)) > 0.1
```

**Lesson:** pg_trgm is case-sensitive. Always use `lower()` for medical text matching where capitalisation is inconsistent (PT names are title-cased, narratives are often lowercase).

---

### Bug #5 — Narrative embedding dilution

**Symptom:** Vector search returned completely unrelated PTs (`"Muscle relaxant drug level"`, `"Pudendal nerve terminal motor latency test"`) for a hypoglycaemia narrative. "Hypoglycaemia" (plain) appeared at vector rank ~631.

**Root cause:**  
PubMedBERT with **mean pooling** over a long narrative produces a vector that is the *average* of all token representations. A narrative mentioning `"insulin pump"`, `"bolus delivery"`, `"blood glucose"`, `"confused"`, and `"hypoglycaemia"` produces a diluted embedding that doesn't strongly represent any single clinical concept.

The diagnose script confirmed:
```
Cosine similarity (stored 'Hypoglycaemia' embedding vs narrative): 0.9104
```
But the TOP vector result `"Drug monitoring procedure incorrectly performed"` had similarity `0.9527`. The similarities are clustered in a narrow range (0.93-0.95) because everything gets "close" in the average embedding space.

**Fix:** Encode only the **first sentence** of the narrative for the vector arm:
```python
first_sentence = query.split(".")[0].strip()
query_embedding = self.model.encode(first_sentence)
```
The first sentence of a MAUDE report typically describes the primary adverse event (`"Patient experienced hypoglycaemia after insulin pump delivered unexpected bolus"`). Shorter text → less dilution → better semantic focus.

**Why not use [CLS] token instead of mean pooling?**  
The [CLS] token in BERT is trained for classification tasks (next sentence prediction), not sentence similarity. Mean pooling is the standard approach for sentence embeddings with BERT-based models.

**Lesson:** Mean pooling over long texts produces diluted embeddings. For retrieval tasks, use the most focused excerpt of the input text — typically the first sentence or a key phrase. This is a fundamental limitation of bi-encoder architectures for long documents.

---

### Bug #6 — IVFFlat probes too low

**Symptom:** Vector search returned different wrong results on each run (non-deterministic).

**Root cause:**  
The IVFFlat index was created with `lists=100` (100 clusters). The default `ivfflat.probes=1` means pgvector searches only the single nearest cluster at query time. With 27k vectors split into 100 clusters (~270 PTs per cluster), searching 1 cluster means examining only ~270 PTs — a 1% sample. This gives fast but highly inaccurate approximate nearest neighbour results.

**Fix:** Set `probes=100` (search all clusters = exact results):
```sql
SET ivfflat.probes = 100;
```
For production with a larger dataset, `probes=10-20` gives a good accuracy/speed trade-off.

**Lesson:** IVFFlat is an *approximate* nearest neighbour index. Always verify index accuracy before trusting search results. The `probes` parameter controls the recall/speed trade-off — never leave it at the default of 1 for production use.

---
## Part 3: Remaining Limitations

The pipeline now passes the smoke test, but two known limitations remain:

### 1. First-sentence heuristic is fragile
Not all MAUDE narratives lead with the primary adverse event. Some start with patient demographics, device information, or context. A more robust approach would use a sentence importance scorer or LLM-based key phrase extraction.

### 2. Vector arm quality for clinical text
PubMedBERT embeddings of short PT names vs. longer narrative text will never have perfect semantic alignment. The CrossEncoder (Stage 2) compensates for this significantly — its joint attention over (query, PT) pairs is much more accurate.

The practical implication: the hybrid search retrieves a candidate set, and the CrossEncoder does the real heavy lifting of identifying the correct PT. The hybrid search just needs to ensure the right PT is somewhere in the Top-20.

### 3. `probes=100` is slow
With `probes=100` we scan all clusters (exact search). For the current 27k PT dataset this is fast (~200ms). For larger datasets, reduce to `probes=10-20` after validating that the target PT appears in the top results.

---
## Part 4: What Each Fix Changed

| Fix | File | Line/Function | Change |
|-----|------|--------------|--------|
| Bug #2 | `hybrid_search.py` | `_vector_search()` | Split `SET` + `SELECT` into two `cur.execute()` calls |
| Bug #3 | `hybrid_search.py` | `_bm25_search()` | `similarity()` → `word_similarity()` |
| Bug #4 | `hybrid_search.py` | `_bm25_search()` | Added `lower()` to both sides of comparison |
| Bug #5 | `hybrid_search.py` | `search()` | Encode only `query.split(".")[0]` for vector arm |
| Bug #6 | `hybrid_search.py` | `_vector_search()` | `SET ivfflat.probes = 100` |
| Bug #1 | Infrastructure | — | `docker stop ny-taxi-db` (local port conflict) |